# Simulation ALS Data: Two-Epoch TAM3C2 + Multi-Scale Analysis

Time-Adaptive M3C2 using `py4dgeo.tam3c2` on one reference epoch and one explicitly selected target epoch.

**Dataset:** 
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als`

The TAM3C2 full-time series is calculated on the basis of all original timestamps, including the 15 missing timestamps.

Standard M3C2 is calculated solely on the basis of the remaining observed epochs.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingAlgorithm, RegionGrowingSeed, temporal_averaging
from py4dgeo.data_loader import read_pc_epochs_and_assign_timestamps


## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled1'

# Two-epoch TAM3C2 setup: one reference epoch and one explicit target epoch.
reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)
target_timestamp = datetime(2020, 1, 22, 0, 0, 0)

# Spatial block selection. Use "all" for the full scene, or 1-4 for one 40x40 m block.
# Full scene x/y range: [-20, 60]. Block edges are x=-20/20/60 and y=-20/20/60.
# Numbering: 1=lower-left, 2=lower-right, 3=upper-left, 4=upper-right.
selected_block = 'all'
scene_x_range = (-20.0, 60.0)
scene_y_range = (-20.0, 60.0)
scene_subset_label = f"block{selected_block}" if selected_block != "all" else "all"

# Generated point clouds are stored beside the source dataset. Analysis archives
# are written to a separate folder in the notebook directory.
notebook_directory = os.getcwd()
dataset_name = os.path.basename(os.path.normpath(data_path))
gap_dataset_name = f"{dataset_name}_temporal_gap"
temporal_gap_data_path = os.path.join(
    os.path.dirname(os.path.normpath(data_path)),
    gap_dataset_name,
)
output_directory = os.path.join(notebook_directory, gap_dataset_name)
os.makedirs(temporal_gap_data_path, exist_ok=True)
os.makedirs(output_directory, exist_ok=True)

reference_file_path = os.path.join(notebook_directory, "simulation2_reference.zip")
output_prefix = f"{gap_dataset_name}_{scene_subset_label}"
single_target_output_path = os.path.join(
    output_directory,
    f"{output_prefix}_single_target_multiscale_tam3c2.zip",
)
best_scale_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_single_target_best_scale_idx{{best_combo_idx}}_weighted_tam3c2.zip",
)
no_weight_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_single_target_best_scale_idx{{best_combo_idx}}_unweighted_tam3c2.zip",
)
temporal_only_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_single_target_best_scale_idx{{best_combo_idx}}_temporal_only_tam3c2.zip",
)
direct_m3c2_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_single_target_withheld_target_oracle_m3c2.zip",
)
timeseries_tam3c2_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_full_timeline_reconstructed_best_scale_idx{{best_combo_idx}}_weighted_tam3c2.zip",
)
timeseries_m3c2_output_template = os.path.join(
    output_directory,
    f"{output_prefix}_observed_epochs_only_best_scale_idx{{best_combo_idx}}_standard_m3c2.zip",
)

# Remove 15 epochs from the 61-day series. The configured target is
# deliberately withheld and must be reconstructed from neighboring observations.
# The remaining indices contain both isolated removals and consecutive runs.
temporal_gap_indices = [2, 3, 7, 18, 19, 21, 23, 24, 25, 38, 39, 47, 48, 49, 56]

# TAM3C2 parameters - multi-scale grid for reference/target-only aggregation
normal_radii = [0.1, 0.2, 0.3, 0.5]
max_window_ratio = [0.2, 0.3, 0.5]
required_points = 10
cyl_radius = 1.0
max_distance = 10.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
include_center_epoch = True
keep_neighborhoods = True

# 4D-OBC parameters
obc_neighborhood_radius = 1.0
obc_min_segments = 10
obc_minperiod = 3
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_seed_subsampling = 1


### corepoints from the reference file

In [ ]:
ref_analysis = py4dgeo.SpatiotemporalAnalysis(reference_file_path, force=False)
all_corepoints = ref_analysis.corepoints.cloud

x_min, x_max = scene_x_range
y_min, y_max = scene_y_range
x_mid = 0.5 * (x_min + x_max)
y_mid = 0.5 * (y_min + y_max)

block_bounds = {
    1: (x_min, x_mid, y_min, y_mid),
    2: (x_mid, x_max, y_min, y_mid),
    3: (x_min, x_mid, y_mid, y_max),
    4: (x_mid, x_max, y_mid, y_max),
}

if selected_block == "all":
    corepoints = all_corepoints
    print(f"Selected full scene: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")
    print(f"Corepoints used: {len(corepoints):,}/{len(all_corepoints):,}")
else:
    if selected_block not in block_bounds:
        raise ValueError(f"selected_block must be 'all' or one of {sorted(block_bounds)}, got {selected_block}")

    block_x_min, block_x_max, block_y_min, block_y_max = block_bounds[selected_block]
    x_in_block = (all_corepoints[:, 0] >= block_x_min) & (
        all_corepoints[:, 0] < block_x_max if selected_block in (1, 3) else all_corepoints[:, 0] <= block_x_max
    )
    y_in_block = (all_corepoints[:, 1] >= block_y_min) & (
        all_corepoints[:, 1] < block_y_max if selected_block in (1, 2) else all_corepoints[:, 1] <= block_y_max
    )
    block_mask = x_in_block & y_in_block

    corepoints = all_corepoints[block_mask]
    if len(corepoints) == 0:
        raise ValueError(
            f"No corepoints found in block {selected_block}: "
            f"x=[{block_x_min}, {block_x_max}], y=[{block_y_min}, {block_y_max}]"
        )

    print(f"Selected block {selected_block}: x=[{block_x_min}, {block_x_max}], y=[{block_y_min}, {block_y_max}]")
    print(f"Corepoints in selected block: {len(corepoints):,}/{len(all_corepoints):,}")


## 2. Load epochs and select the reference/target pair

In [ ]:
epochs_daily = sorted(
    read_pc_epochs_and_assign_timestamps(
        folder=data_path,
        start_time=datetime(2020, 1, 1),
        time_increment=timedelta(days=1),
    ),
    key=lambda epoch: epoch.timestamp,
)

if len(temporal_gap_indices) != 15 or len(set(temporal_gap_indices)) != 15:
    raise ValueError("temporal_gap_indices must contain 15 unique indices")
if min(temporal_gap_indices) < 0 or max(temporal_gap_indices) >= len(epochs_daily):
    raise ValueError("A temporal-gap index is outside the loaded time series")

gap_index_set = set(temporal_gap_indices)
temporal_gap_timestamps = {
    epochs_daily[index].timestamp for index in gap_index_set
}
if reference_timestamp in temporal_gap_timestamps:
    raise ValueError("The reference epoch must remain observed")
if target_timestamp not in temporal_gap_timestamps:
    raise ValueError("The configured target epoch must be included in the temporal gap")

# Only these epochs are available to TAM3C2 as aggregation support and to
# standard M3C2 as observed target epochs.
epochs_all = [
    epoch for index, epoch in enumerate(epochs_daily)
    if index not in gap_index_set
]
removed_epochs = [epochs_daily[index] for index in sorted(gap_index_set)]

reference_epoch = next(
    (epoch for epoch in epochs_all if epoch.timestamp == reference_timestamp),
    None,
)
# This object is withheld from epochs_all. TAM3C2 uses its timestamp to define
# the reconstruction time; its point cloud is never used for aggregation.
target_epoch = next(
    (epoch for epoch in epochs_daily if epoch.timestamp == target_timestamp),
    None,
)
if reference_epoch is None or target_epoch is None:
    raise ValueError("The configured reference or target timestamp is absent")
if any(epoch.timestamp == target_timestamp for epoch in epochs_all):
    raise RuntimeError("Target-data leakage: target epoch is still present in epochs_all")

# Keep the persisted folder synchronized with the actually observed epochs.
for filename in os.listdir(temporal_gap_data_path):
    if filename.startswith("epoch_") and filename.lower().endswith(".xyz"):
        os.remove(os.path.join(temporal_gap_data_path, filename))

for original_index, epoch in enumerate(epochs_daily):
    if original_index in gap_index_set:
        continue
    timestamp_suffix = epoch.timestamp.strftime("%Y%m%dT%H%M%S")
    filename = f"epoch_{original_index:03d}_{timestamp_suffix}.xyz"
    np.savetxt(os.path.join(temporal_gap_data_path, filename), epoch.cloud, fmt="%.8f")

print(f"Loaded daily epochs: {len(epochs_daily)}")
print(f"Removed epochs: {len(removed_epochs)}; observed epochs: {len(epochs_all)}")
print(f"Removed timestamps: {[epoch.timestamp for epoch in removed_epochs]}")
print(f"Withheld evaluation target: {target_epoch.timestamp}")
print(f"Target present in aggregation support: {any(e.timestamp == target_timestamp for e in epochs_all)}")
print(f"Saved the observed temporal-gap series to: {temporal_gap_data_path}")


In [ ]:
print(f"Reference epoch (observed): {reference_epoch.timestamp}")
print(f"Evaluation target (withheld): {target_epoch.timestamp}")


In [ ]:
print("Irregular temporal gaps (removed timestamps):")
for epoch in removed_epochs:
    print(f"  {epoch.timestamp}")


## timeline figure

In [ ]:
# Plot retained sampled timestamps and original epoch indices on a time axis.
import matplotlib.dates as mdates
sampled_indices = [
    index
    for index, epoch in enumerate(epochs_daily)
    if index not in set(temporal_gap_indices)
]
sampled_timestamps = [
    epochs_daily[index].timestamp
    for index in sampled_indices
]

removed_indices = sorted(temporal_gap_indices)
removed_timestamps = [
    epochs_daily[index].timestamp
    for index in removed_indices
]

fig, ax = plt.subplots(figsize=(18, 4.5))

# Full original daily timeline.
ax.plot(
    [epoch.timestamp for epoch in epochs_daily],
    np.ones(len(epochs_daily)),
    color="lightgray",
    linewidth=1.5,
    zorder=1,
    label="Original daily timeline",
)

# Retained / sampled epochs.
ax.scatter(
    sampled_timestamps,
    np.ones(len(sampled_timestamps)),
    s=70,
    color="tab:blue",
    edgecolor="black",
    linewidth=0.5,
    zorder=3,
    label=f"Retained sampled epochs ({len(sampled_timestamps)})",
)

# Removed temporal-gap epochs.
ax.scatter(
    removed_timestamps,
    np.full(len(removed_timestamps), 0.88),
    s=80,
    marker="x",
    color="tab:red",
    linewidth=2,
    zorder=4,
    label=f"Removed gap epochs ({len(removed_timestamps)})",
)

# Mark reference and target epochs.
ax.scatter(
    [reference_timestamp],
    [1.12],
    s=130,
    marker="*",
    color="tab:green",
    edgecolor="black",
    linewidth=0.5,
    zorder=5,
    label="Reference epoch",
)
ax.scatter(
    [target_timestamp],
    [1.12],
    s=130,
    marker="*",
    color="tab:orange",
    edgecolor="black",
    linewidth=0.5,
    zorder=5,
    label="Target epoch",
)

# Annotate retained sampled epochs with original index and timestamp.
for index, timestamp in zip(sampled_indices, sampled_timestamps):
    ax.text(
        timestamp,
        1.04,
        f"{index}\n{timestamp:%m-%d}",
        ha="center",
        va="bottom",
        fontsize=7,
        rotation=90,
    )

# Annotate removed epochs with original index.
for index, timestamp in zip(removed_indices, removed_timestamps):
    ax.text(
        timestamp,
        0.80,
        f"{index}",
        ha="center",
        va="top",
        fontsize=8,
        color="tab:red",
    )

ax.set_title("Temporal-gap time series: sampled timestamps and original epoch indices")
ax.set_xlabel("Timestamp")
ax.set_yticks([0.88, 1.00, 1.12])
ax.set_yticklabels(["Removed", "Sampled", "Reference / target"])
ax.set_ylim(0.72, 1.24)

ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

ax.grid(axis="x", which="major", linestyle="--", alpha=0.45)
ax.grid(axis="x", which="minor", linestyle=":", alpha=0.2)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.4), ncol=4)

plt.xticks(rotation=45, ha="right")
plt.subplots_adjust(bottom=0.32)
plt.show()

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

single_target_multiscale_analysis = py4dgeo.SpatiotemporalAnalysis(single_target_output_path, force=True)
single_target_multiscale_analysis.reference_epoch = reference_epoch
single_target_multiscale_analysis.corepoints = corepoints
single_target_multiscale_analysis.m3c2 = tam

single_target_multiscale_analysis.add_epochs(target_epoch)
print(f"target epoch: {target_epoch.timestamp}")
print(f"epochs_timeseries used by TAM3C2: {[e.timestamp for e in tam.epochs_timeseries]}")
print(f"include_center_epoch: {tam.include_center_epoch}")
print(f"distances shape: {single_target_multiscale_analysis.distances.shape}")
print(f"uncertainties shape: {single_target_multiscale_analysis.uncertainties.shape}")


## 4. statistic aggregation diagnostics for the selected target

In [ ]:
diag = tam.diagnostics()


In [ ]:
diag['scale_idx'] # multiscale index of the scale used for each corepoint


In [ ]:
combinations = tam._scale_combinations 

i = 0  
idx = tam._opt_scale_idx[i]
print(f"corepoint {i} scale: {combinations[idx]}")


In [ ]:
diag['window_used_ref'] # unit = seconds, the time window used for aggregation, 86400=1 day, 最远 Epoch”距离中心点的时间跨度


In [ ]:
diag['window_used_ref'][0]


In [ ]:
diag['n_after_ref'] # number of epochs after reference aggregation


In [ ]:
diag['n_before_ref']


In [ ]:
diag['n_points_ref']


In [ ]:
tam._neighborhoods[0]


## visual analysis of aggregation

In [ ]:
# Visual diagnostics for temporal aggregation around each corepoint.
# Requires `tam`, `diag`, `reference_epoch`, `target_epoch`, and `corepoints` from previous cells.

def _diag_col(name, col=0):
    value = diag[name]
    arr = np.asarray(value)
    return arr[:, col] if arr.ndim == 2 else arr


def _plot_cp_map(ax, values, title, cmap='viridis', s=3, vmin=None, vmax=None, discrete=False):
    sc = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=values, s=s, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.axis('equal')
    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    if discrete:
        vals = np.unique(np.asarray(values)[np.isfinite(values)])
        if len(vals) <= 12:
            cbar.set_ticks(vals)
    return sc


def _count_cylinder_points_in_epoch(cp, normal, epoch_idx, cyl_radius, max_distance, fallback_epoch=None):
    bounding_r = np.sqrt(cyl_radius * cyl_radius + max_distance * max_distance)
    if epoch_idx is None:
        if fallback_epoch is None:
            return 0
        # Build a temporary kdtree for the fallback epoch
        from scipy.spatial import cKDTree
        tree = cKDTree(fallback_epoch.cloud)
        idxs = tree.query_ball_point(cp, bounding_r)
        pts_source = fallback_epoch.cloud
    else:
        idxs = tam._kdtrees[epoch_idx].query_ball_point(cp, bounding_r)
        pts_source = tam.epochs_timeseries[epoch_idx].cloud
    if not idxs:
        return 0
    pts = pts_source[idxs]
    vec = pts - cp
    along = vec @ normal
    perp_sq = np.einsum('ij,ij->i', vec, vec) - along * along
    mask = (perp_sq <= cyl_radius * cyl_radius) & (np.abs(along) <= max_distance)
    return int(np.count_nonzero(mask))


tam._build_index()
target_col = 0
scale_idx = np.asarray(diag['scale_idx']).astype(int)
combos = tam._scale_combinations

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
tgt_idx_in_ts = tam._find_epoch_index(target_epoch)
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts

n_points_ref_after = _diag_col('n_points_ref', target_col)
n_points_tgt_after = _diag_col('n_points_tgt', target_col)

# Exact number of contributing epochs if keep_neighborhoods=True; fallback to diagnostics otherwise.
nbhd = tam._neighborhoods[target_col] if tam._neighborhoods is not None else None
if nbhd is not None:
    n_epochs_ref = np.array([
        len(np.unique(record['ref_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
    n_epochs_tgt = np.array([
        len(np.unique(record['tgt_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
else:
    n_epochs_ref = _diag_col('n_before_ref', target_col) + _diag_col('n_after_ref', target_col)
    n_epochs_tgt = _diag_col('n_before_tgt', target_col) + _diag_col('n_after_tgt', target_col)
    print('keep_neighborhoods=False: epoch-count maps use n_before + n_after diagnostics.')

# Center-epoch-only cylinder counts before temporal aggregation.
n_points_ref_before = np.zeros(len(corepoints), dtype=int)
n_points_tgt_before = np.zeros(len(corepoints), dtype=int)
planarity_selected = np.full(len(corepoints), np.nan)

for i, cp in enumerate(corepoints):
    normal = tam._ref_normals[i]
    sr, wr = combos[scale_idx[i]]

    n_points_ref_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, ref_idx_in_ts, tam.cyl_radius, tam.max_distance
    )
    n_points_tgt_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, tgt_idx_in_ts, tam.cyl_radius, tam.max_distance
    )

    pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, time_range * wr)
    if pts is not None and len(pts) >= 3:
        planarity_selected[i], _ = tam._planarity_and_normal(pts)

# Required-points status: 0=neither side, 1=ref only, 2=target only, 3=both sides.
required_before = (
    (n_points_ref_before >= required_points).astype(int)
    + 2 * (n_points_tgt_before >= required_points).astype(int)
)
required_after = (
    (n_points_ref_after >= required_points).astype(int)
    + 2 * (n_points_tgt_after >= required_points).astype(int)
)

print('Scale index -> (normal_radius, max_window_ratio)')
for idx, combo in enumerate(combos):
    print(f'  {idx}: {combo}')
print('Required status code: 0=neither side, 1=ref only, 2=target only, 3=both sides')

# 1. Aggregation epoch-count maps.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
vmax_epochs = max(np.nanmax(n_epochs_ref), np.nanmax(n_epochs_tgt))
_plot_cp_map(axs[0], n_epochs_ref, 'Reference aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
_plot_cp_map(axs[1], n_epochs_tgt, 'Target aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
plt.tight_layout()
plt.show()

# 2. Point-count maps before vs after temporal aggregation.
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
vmax_points = np.nanpercentile(
    np.r_[n_points_ref_before, n_points_tgt_before, n_points_ref_after, n_points_tgt_after],
    98,
)
_plot_cp_map(axs[0, 0], n_points_ref_before, 'Reference center epoch: cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[0, 1], n_points_ref_after, 'Reference after temporal aggregation: points used', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 0], n_points_tgt_before, 'Withheld target: observed cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 1], n_points_tgt_after, 'Target after temporal aggregation: points used', 'magma', vmax=vmax_points)
plt.tight_layout()
plt.show()

# 3. Required-points status before and after aggregation.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
_plot_cp_map(axs[0], required_before, f'Before aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
_plot_cp_map(axs[1], required_after, f'After aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
plt.tight_layout()
plt.show()

# 4. Selected spatiotemporal scale and planarity.
# Bivariate color mixture: blue = selected normal radius, red = selected max_window_ratio.
selected_radius = np.array([combos[idx][0] for idx in scale_idx], dtype=float)
selected_window_ratio = np.array([combos[idx][1] for idx in scale_idx], dtype=float)


def _normalize_to_unit(values):
    values = np.asarray(values, dtype=float)
    scaled = np.zeros_like(values, dtype=float)
    finite = np.isfinite(values)
    if not np.any(finite):
        return scaled, np.nan, np.nan
    vmin = float(np.nanmin(values[finite]))
    vmax = float(np.nanmax(values[finite]))
    if vmax > vmin:
        scaled[finite] = (values[finite] - vmin) / (vmax - vmin)
    else:
        scaled[finite] = 0.5
    scaled[~finite] = np.nan
    return scaled, vmin, vmax


def _mix_scale_colors(radius_level, ratio_level):
    radius_level = np.asarray(radius_level, dtype=float)
    ratio_level = np.asarray(ratio_level, dtype=float)
    low = np.array([0.93, 0.93, 0.93])
    blue = np.array([0.08, 0.30, 0.95])
    red = np.array([0.95, 0.08, 0.28])
    purple = np.array([0.28, 0.00, 0.45])

    radius = np.nan_to_num(radius_level, nan=0.0)[..., None]
    ratio = np.nan_to_num(ratio_level, nan=0.0)[..., None]
    return (
        (1.0 - radius) * (1.0 - ratio) * low
        + radius * (1.0 - ratio) * blue
        + (1.0 - radius) * ratio * red
        + radius * ratio * purple
    )


radius_level, radius_min, radius_max = _normalize_to_unit(selected_radius)
ratio_level, ratio_min, ratio_max = _normalize_to_unit(selected_window_ratio)
mixed_colors = _mix_scale_colors(radius_level, ratio_level)

fig = plt.figure(figsize=(17, 5.8), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.55, 1.15])
ax_scale = fig.add_subplot(gs[0, 0])
ax_key = fig.add_subplot(gs[0, 1])
ax_planarity = fig.add_subplot(gs[0, 2])

ax_scale.scatter(
    corepoints[:, 0],
    corepoints[:, 1],
    c=mixed_colors,
    s=6,
    linewidths=0,
    alpha=0.9,
)
ax_scale.set_title('Selected spatiotemporal scale')
ax_scale.set_xlabel('X [m]')
ax_scale.set_ylabel('Y [m]')
ax_scale.set_aspect('equal', adjustable='box')

key_steps = 120
radius_grid = np.linspace(0.0, 1.0, key_steps)
ratio_grid = np.linspace(0.0, 1.0, key_steps)
R, T = np.meshgrid(radius_grid, ratio_grid)
key_colors = _mix_scale_colors(R, T)
ax_key.imshow(
    key_colors,
    origin='lower',
    extent=[radius_min, radius_max, ratio_min, ratio_max],
    aspect='auto',
)
ax_key.set_title('Color mixture')
ax_key.set_xlabel('normal radius [m]')
ax_key.set_ylabel('max_window_ratio')
ax_key.set_xticks(sorted(set(selected_radius)))
ax_key.set_yticks(sorted(set(selected_window_ratio)))
ax_key.tick_params(labelsize=8)

_plot_cp_map(ax_planarity, planarity_selected, 'Planarity at selected scale', 'viridis')
plt.show()


## 5. Multi-scale analysis 

In [ ]:
# Full-corepoint multi-scale planarity map on the (normal_radii x max_window_ratio) grid.
#
# Important: the mean-planarity panel below does NOT run full TAM3C2 once per
# combo. It recomputes the spherical normal-estimation neighborhood for every
# (scale combo, corepoint) pair and reports the PCA planarity. This is the same
# criterion used by TAM3C2's scale-selection contest.
#
# The selection-frequency panel comes from the already-run multiscale TAM3C2
# cache: each corepoint votes for the combo selected by TAM3C2.
#
# The selected best single-scale combo is then run once as a separate TAM3C2
# single_target_multiscale_analysis and saved for later comparison.

radii  = list(tam.normal_radii)     if hasattr(tam.normal_radii,     '__iter__') else [tam.normal_radii]
ratios = list(tam.max_window_ratio) if hasattr(tam.max_window_ratio, '__iter__') else [tam.max_window_ratio]
combos = tam._scale_combinations    # ordered: for sr in radii: for wr in ratios

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0

# --- 1. planarity for every (combo, corepoint), no subsampling --------------
n_corepoints = len(corepoints)
planarity = np.full((len(combos), n_corepoints), np.nan)

for k, (sr, wr) in enumerate(combos):
    mw = time_range * wr
    print(f"Computing planarity for combo {k}/{len(combos)-1}: normal_radius={sr:g}, max_window_ratio={wr:g}")
    for i, cp in enumerate(corepoints):
        pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, mw)
        if pts is None or len(pts) < 3:
            continue
        pl, _ = tam._planarity_and_normal(pts)
        planarity[k, i] = pl

mean_pl_flat = np.nanmean(planarity, axis=1)
mean_pl = mean_pl_flat.reshape(len(radii), len(ratios))

# --- 2. selection frequency from cached multiscale diagnostics --------------
diag = tam.diagnostics()
scale_idx_all = np.asarray(diag['scale_idx']).astype(int)
counts = np.bincount(scale_idx_all, minlength=len(combos))
freq_flat = counts / counts.sum()
freq = freq_flat.reshape(len(radii), len(ratios))
combo_idx_grid = np.arange(len(combos)).reshape(len(radii), len(ratios))

# --- 3. choose one best combo and save a single-scale TAM3C2 result ----------
best_mean_planarity_idx = int(np.nanargmax(mean_pl_flat))
most_selected_idx = int(np.argmax(counts))

# Default strategy: choose by mean planarity. If several combinations are tied
# within this tolerance, prefer the smaller spatial radius and then the smaller
# temporal window ratio.
best_combo_strategy = "mean_planarity" #！！！
planarity_tie_tolerance = 1e-3

if best_combo_strategy == "mean_planarity":
    best_mean_planarity = float(np.nanmax(mean_pl_flat))
    tied_mean_planarity_idx = np.flatnonzero(
        np.isfinite(mean_pl_flat)
        & (mean_pl_flat >= best_mean_planarity - planarity_tie_tolerance)
    )
    if len(tied_mean_planarity_idx) == 0:
        raise ValueError("No finite mean planarity value found.")
    best_combo_idx = int(
        min(
            tied_mean_planarity_idx,
            key=lambda idx: (combos[int(idx)][0], combos[int(idx)][1]),
        )
    )
elif best_combo_strategy == "most_selected":
    best_combo_idx = most_selected_idx
else:
    raise ValueError(f"Unknown best_combo_strategy: {best_combo_strategy!r}")

best_sr, best_wr = combos[best_combo_idx]
timeseries_tam3c2_output_path = timeseries_tam3c2_output_template.format(
    best_combo_idx=best_combo_idx
)
timeseries_m3c2_output_path = timeseries_m3c2_output_template.format(
    best_combo_idx=best_combo_idx
)
best_scale_output_path = best_scale_output_template.format(
    best_combo_idx=best_combo_idx
)

print("\nScale index -> (normal_radius, max_window_ratio)")
for idx, combo in enumerate(combos):
    print(f"  {idx}: {combo}")
print(f"\nBest by mean planarity : idx={best_mean_planarity_idx}, combo={combos[best_mean_planarity_idx]}, mean_planarity={mean_pl_flat[best_mean_planarity_idx]:.4f}")
print(f"Best by selection freq : idx={most_selected_idx}, combo={combos[most_selected_idx]}, frequency={freq_flat[most_selected_idx]:.1%}")
if best_combo_strategy == "mean_planarity":
    print(
        "Mean-planarity tie candidates "
        f"(within {planarity_tie_tolerance:g}): "
        f"{[(int(idx), combos[int(idx)], float(mean_pl_flat[int(idx)])) for idx in tied_mean_planarity_idx]}"
    )
print(f"Using best_combo_strategy={best_combo_strategy!r}: idx={best_combo_idx}, combo=({best_sr}, {best_wr})")

best_scale_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

best_scale_analysis = py4dgeo.SpatiotemporalAnalysis(best_scale_output_path, force=True)
best_scale_analysis.reference_epoch = reference_epoch
best_scale_analysis.corepoints = corepoints
best_scale_analysis.m3c2 = best_scale_tam
best_scale_analysis.add_epochs(target_epoch)

multi_dist = single_target_multiscale_analysis.distances[:, 0]
best_dist = best_scale_analysis.distances[:, 0]
compare_valid = np.isfinite(multi_dist) & np.isfinite(best_dist)
print(f"Saved best single-scale TAM3C2 single_target_multiscale_analysis: {best_scale_output_path}")
print(f"Best single-scale distances shape: {best_scale_analysis.distances.shape}")
print(f"Compared to per-corepoint multiscale TAM3C2 on {compare_valid.sum()} valid corepoints:")
print(f"  mean(best - multiscale) = {np.nanmean(best_dist[compare_valid] - multi_dist[compare_valid]):.4f} m")
print(f"  MAE(best vs multiscale) = {np.nanmean(np.abs(best_dist[compare_valid] - multi_dist[compare_valid])):.4f} m")

# --- 4. two-panel heatmap with combo index labels ---------------------------
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, M, title, cmap, fmt in [
    (axs[0], mean_pl, 'Mean planarity over all corepoints', 'viridis', '.3f'),
    (axs[1], freq,    'Selection frequency of winning scale', 'magma', '.1%'),
]:
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(ratios)))
    ax.set_xticklabels([f'{r:g}' for r in ratios])
    ax.set_yticks(range(len(radii)))
    ax.set_yticklabels([f'{r:g}' for r in radii])
    ax.set_xlabel('max_window_ratio')
    ax.set_ylabel('normal_radii [m]')
    ax.set_title(title)
    for row in range(M.shape[0]):
        for col in range(M.shape[1]):
            val = M[row, col]
            if not np.isnan(val):
                ax.text(
                    col,
                    row,
                    format(val, fmt),
                    ha='center',
                    va='center',
                    color='white',
                    fontsize=9,
                )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axs:
    best_row, best_col = np.argwhere(combo_idx_grid == best_combo_idx)[0]
    ax.scatter(best_col, best_row, s=220, facecolors='none', edgecolors='cyan', linewidths=2.5)

plt.tight_layout()
plt.show()


## 7. Compare against reference distances from `simulation2_reference.zip`

Load the precomputed analysis archive, select the column matching the current target epoch, align corepoints, and compare the current TAM3C2 distance estimates against the reference distances.

In [ ]:
ref_analysis.smoothed_distances


In [ ]:
indices = np.nonzero(ref_analysis.distances)[0]
np.unique(indices)


In [ ]:
# Recompute and save the no-weight best single-scale TAM3C2 single_target_multiscale_analysis.
# This uses the best scale selected in the previous multiscale-single_target_multiscale_analysis cell,
# but turns temporal weighting off for both reference and target aggregation.

no_weight_output_path = no_weight_output_template.format(
    best_combo_idx=best_combo_idx
)

no_weight_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=Weighting.NONE,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

no_weight_analysis = py4dgeo.SpatiotemporalAnalysis(no_weight_output_path, force=True)
no_weight_analysis.reference_epoch = reference_epoch
no_weight_analysis.corepoints = corepoints
no_weight_analysis.m3c2 = no_weight_tam
no_weight_analysis.add_epochs(target_epoch)


print(f"Saved unweighted best-scale TAM3C2 single_target_multiscale_analysis: {no_weight_output_path}")
print(f"Unweighted distances shape: {no_weight_analysis.distances.shape}")
print(f"Unweighted uncertainties shape: {no_weight_analysis.uncertainties.shape}")


In [ ]:
# Recompute and save the temporal-only weighted best single-scale TAM3C2 single_target_multiscale_analysis.
# This keeps the Gaussian temporal weighting but disables the spatial part of
# the weighting kernel, so auxiliary epochs are down-weighted only by time.

temporal_only_output_path = temporal_only_output_template.format(
    best_combo_idx=best_combo_idx
)

temporal_only_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    spatial_weighting=False,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

temporal_only_analysis = py4dgeo.SpatiotemporalAnalysis(temporal_only_output_path, force=True)
temporal_only_analysis.reference_epoch = reference_epoch
temporal_only_analysis.corepoints = corepoints
temporal_only_analysis.m3c2 = temporal_only_tam
temporal_only_analysis.add_epochs(target_epoch)

print(f"Saved temporal-only weighted best-scale TAM3C2 single_target_multiscale_analysis: {temporal_only_output_path}")
print(f"Temporal-only weighted distances shape: {temporal_only_analysis.distances.shape}")
print(f"Temporal-only weighted uncertainties shape: {temporal_only_analysis.uncertainties.shape}")


In [ ]:
# Standard M3C2 cannot estimate this target because its point cloud was withheld.
# Running M3C2(reference_epoch, target_epoch) here would leak the hidden target
# observations and would constitute an oracle experiment, not a temporal-gap baseline.
direct_m3c2_analysis = None
print(
    "Direct M3C2 at the withheld target: unavailable "
    "(no target point cloud in the observed temporal-gap dataset)."
)


## result

In [ ]:
# Plot five error maps and five roughness maps.
#
# Error is defined as: method distance - mesh-reference distance.
# Roughness is represented by the mean M3C2 spread per corepoint:
# 0.5 * (spread1 + spread2), where spread1 is the reference-side spread and
# spread2 is the target-side spread.

from scipy.spatial import cKDTree


def _target_column_from_reference(reference_analysis, target_time, reference_time):
    expected_delta = target_time - reference_time
    timedeltas = list(reference_analysis.timedeltas)
    matches = [idx for idx, delta in enumerate(timedeltas) if delta == expected_delta]
    if matches:
        return matches[0]

    delta_seconds = np.array([abs((delta - expected_delta).total_seconds()) for delta in timedeltas])
    nearest = int(np.argmin(delta_seconds))
    print(
        f"No exact reference timedelta match for {expected_delta}; "
        f"using nearest column {nearest} ({timedeltas[nearest]})."
    )
    return nearest


def _aligned_reference_distance(reference_analysis, query_corepoints, target_time, reference_time):
    target_col = _target_column_from_reference(reference_analysis, target_time, reference_time)
    reference_corepoints = reference_analysis.corepoints.cloud
    reference_distance_all = reference_analysis.distances[:, target_col]

    if len(reference_corepoints) == len(query_corepoints) and np.allclose(reference_corepoints, query_corepoints):
        return reference_distance_all, np.arange(len(query_corepoints)), target_col

    tree = cKDTree(reference_corepoints[:, :3])
    distances_to_ref, reference_idx = tree.query(query_corepoints[:, :3], k=1)
    print(
        "Corepoints are not identical; using nearest reference corepoint alignment. "
        f"median distance={np.median(distances_to_ref):.4f} m, "
        f"max={np.max(distances_to_ref):.4f} m"
    )
    return reference_distance_all[reference_idx], reference_idx, target_col


def _distance_column(st_analysis, label):
    if st_analysis.distances is None or st_analysis.distances.shape[1] == 0:
        raise ValueError(f"{label} has no distance column")
    return st_analysis.distances[:, 0].astype(float)


def _roughness_from_uncertainty(st_analysis, label):
    if st_analysis.uncertainties is None:
        raise ValueError(f"{label} has no uncertainty array")

    uncertainty = st_analysis.uncertainties[:, 0]
    names = uncertainty.dtype.names or ()
    if "spread1" not in names or "spread2" not in names:
        raise ValueError(f"{label} uncertainty does not contain spread1/spread2 fields: {names}")

    return 0.5 * (uncertainty["spread1"].astype(float) + uncertainty["spread2"].astype(float))


def _finite_percentile(values, percentile, default=1.0):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return default
    limit = float(np.nanpercentile(finite, percentile))
    if not np.isfinite(limit) or limit == 0:
        return default
    return limit


def _format_map_axes(ax):
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect("equal", adjustable="box")


reference_distance, reference_indices, reference_target_col = _aligned_reference_distance(
    ref_analysis,
    corepoints,
    target_timestamp,
    reference_timestamp,
)
print(f"Using mesh-derived reference distance column {reference_target_col} for withheld target {target_timestamp}")
print("Direct M3C2 completeness at this target: 0% (target observation unavailable)")

comparison_results = {
    "Best scale unweighted": {
        "distance": _distance_column(no_weight_analysis, "Best scale unweighted"),
        "roughness": _roughness_from_uncertainty(no_weight_analysis, "Best scale unweighted"),
    },
    "Best scale temporal-only": {
        "distance": _distance_column(temporal_only_analysis, "Best scale temporal-only"),
        "roughness": _roughness_from_uncertainty(temporal_only_analysis, "Best scale temporal-only"),
    },
    "Multiscale weighted": {
        "distance": _distance_column(single_target_multiscale_analysis, "Multiscale weighted"),
        "roughness": _roughness_from_uncertainty(single_target_multiscale_analysis, "Multiscale weighted"),
    },
    "Best scale weighted": {
        "distance": _distance_column(best_scale_analysis, "Best scale weighted"),
        "roughness": _roughness_from_uncertainty(best_scale_analysis, "Best scale weighted"),
    },
}

for label, result in comparison_results.items():
    result["error"] = result["distance"] - reference_distance
    valid_error = np.isfinite(result["error"])
    valid_roughness = np.isfinite(result["roughness"])
    error_rmse = np.sqrt(np.nanmean(result["error"] ** 2))
    error_std = np.nanstd(result["error"])
    print(
        f"{label}: valid error={valid_error.sum():,}/{len(valid_error):,}, "
        f"mean error={np.nanmean(result['error']):.4f} m, "
        f"MAE={np.nanmean(np.abs(result['error'])):.4f} m, "
        f"RMSE={error_rmse:.4f} m, "
        f"STD={error_std:.4f} m, "
        f"mean roughness={np.nanmean(result['roughness']):.4f} m "
        f"({valid_roughness.sum():,} valid)"
    )

common_valid_error = np.logical_and.reduce([
    np.isfinite(result["error"])
    for result in comparison_results.values()
])
print(
    f"\nCommon valid-error corepoints: "
    f"{common_valid_error.sum():,}/{len(common_valid_error):,} "
    "valid for every method and the reference"
)
for label, result in comparison_results.items():
    common_error = result["error"][common_valid_error]
    print(
        f"{label} (common valid): "
        f"mean error={np.mean(common_error):.4f} m, "
        f"MAE={np.mean(np.abs(common_error)):.4f} m, "
        f"RMSE={np.sqrt(np.mean(common_error ** 2)):.4f} m, "
        f"STD={np.std(common_error):.4f} m"
    )

xy = corepoints[:, :2]
error_values = np.concatenate([result["error"] for result in comparison_results.values()])
roughness_values = np.concatenate([result["roughness"] for result in comparison_results.values()])
error_limit = _finite_percentile(np.abs(error_values), 98, default=0.1)
roughness_limit = _finite_percentile(roughness_values, 98, default=0.1)

fig, axs = plt.subplots(1, len(comparison_results), figsize=(27, 5), constrained_layout=True)
for ax, (label, result) in zip(axs, comparison_results.items()):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["error"],
        s=3,
        cmap="seismic_r",
        vmin=-error_limit,
        vmax=error_limit,
    )
    ax.set_title(f"{label}", fontsize=13)
    _format_map_axes(ax)
    fig.colorbar(scatter, ax=ax, label="Error [m]", fraction=0.046, pad=0.04)
plt.show()

fig, axs = plt.subplots(1, len(comparison_results), figsize=(27, 5), constrained_layout=True)
for ax, (label, result) in zip(axs, comparison_results.items()):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["roughness"],
        s=3,
        cmap="viridis",
        vmin=0,
        vmax=roughness_limit,
    )
    ax.set_title(f"{label}\nroughness = mean(spread1, spread2)", fontsize=13)
    _format_map_axes(ax)
    fig.colorbar(scatter, ax=ax, label="Spread roughness [m]", fraction=0.046, pad=0.04)
plt.show()


## Full gap time series and 4D-OBC extraction


In [ ]:
# Run the complete temporal-gap experiment using the selected single scale.
# TAM3C2 receives only epochs_all as point-cloud support, but is evaluated at
# every original timestamp except the fixed reference timestamp.
timeseries_tam3c2 = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

timeseries_tam3c2_analysis = py4dgeo.SpatiotemporalAnalysis(
    timeseries_tam3c2_output_path,
    force=True,
)
timeseries_tam3c2_analysis.reference_epoch = reference_epoch
timeseries_tam3c2_analysis.corepoints = corepoints
timeseries_tam3c2_analysis.m3c2 = timeseries_tam3c2

# Epoch objects at removed timestamps act only as timestamp carriers here.
# Their clouds are absent from timeseries_tam3c2.epochs_timeseries and therefore
# cannot contribute to a TAM3C2 neighborhood.
tam3c2_evaluation_epochs = [
    epoch for epoch in epochs_daily
    if epoch.timestamp != reference_timestamp
]
timeseries_tam3c2_analysis.add_epochs(*tam3c2_evaluation_epochs)

# Standard M3C2 is restricted to timestamps with an observed target point cloud.
m3c2_observed_target_epochs = [
    epoch for epoch in epochs_all
    if epoch.timestamp != reference_timestamp
]
if not m3c2_observed_target_epochs:
    raise ValueError("No observed target epoch is available for standard M3C2")

timeseries_m3c2 = py4dgeo.M3C2(
    epochs=(reference_epoch, m3c2_observed_target_epochs[0]),
    corepoints=corepoints,
    normal_radii=[float(best_sr)],
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

timeseries_m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(
    timeseries_m3c2_output_path,
    force=True,
)
timeseries_m3c2_analysis.reference_epoch = reference_epoch
timeseries_m3c2_analysis.corepoints = corepoints
timeseries_m3c2_analysis.m3c2 = timeseries_m3c2
timeseries_m3c2_analysis.add_epochs(*m3c2_observed_target_epochs)

print(f"Best single scale: normal_radius={best_sr}, max_window_ratio={best_wr}")
print(
    "Shared spatial parameters: "
    f"cyl_radius={cyl_radius}, max_distance={max_distance}, "
    f"registration_error={registration_error}"
)
print(f"Observed support epochs: {len(epochs_all)} of {len(epochs_daily)}")
print(f"TAM3C2 evaluation timestamps: {len(tam3c2_evaluation_epochs)}")
print(f"M3C2 observed target timestamps: {len(m3c2_observed_target_epochs)}")
print(f"Saved reconstructed TAM3C2 series: {timeseries_tam3c2_output_path}")
print(f"Saved observed-only M3C2 series: {timeseries_m3c2_output_path}")
print(f"TAM3C2 distances shape: {timeseries_tam3c2_analysis.distances.shape}")
print(f"Standard M3C2 distances shape: {timeseries_m3c2_analysis.distances.shape}")


In [ ]:
# Extract 4D-OBCs with identical segmentation parameters. TAM3C2 covers the
# reconstructed full timeline; M3C2 covers only observed timestamps.
def create_obc_algorithm():
    return RegionGrowingAlgorithm(
        neighborhood_radius=obc_neighborhood_radius,
        min_segments=obc_min_segments,
        minperiod=obc_minperiod,
        height_threshold=obc_height_threshold,
        thresholds=obc_thresholds,
        seed_subsampling=obc_seed_subsampling,
    )


timeseries_m3c2_analysis.invalidate_results(seeds=True, objects=True)
m3c2_objects = create_obc_algorithm().run(timeseries_m3c2_analysis)
print(
    f"Extracted {len(m3c2_objects)} 4D-OBCs from "
    f"{len(timeseries_m3c2_analysis.seeds)} standard-M3C2 seeds"
)

timeseries_tam3c2_analysis.invalidate_results(seeds=True, objects=True)
tam3c2_objects = create_obc_algorithm().run(timeseries_tam3c2_analysis)
print(
    f"Extracted {len(tam3c2_objects)} 4D-OBCs from "
    f"{len(timeseries_tam3c2_analysis.seeds)} TAM3C2 seeds"
)
